____
# Preprocessed SWOT 250m

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250m, add_mask_inside_swot, build_swath_polygon, add_grid_metrics

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

In [2]:
if True:
    from dask.distributed import Client
    from dask_jobqueue import PBSCluster
    from dask import config

    config.set({"distributed.comm.timeouts.connect": "200s"})
    cluster = PBSCluster(cores=28, processes=28, walltime="01:00:00")
    # cluster = PBSCluster(cores=20, processes=20, walltime='02:00:00')#8
    w = cluster.scale(jobs=2)
else:
    from dask.distributed import Client, LocalCluster

    cluster = LocalCluster()

client = Client(cluster)
client

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'distributed.scheduler.transition-log-length' has been deprecated; please use 'distributed.admin.low-level-log-length' instead
  warnings.warn(
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'distributed.comm.recent-messages-log-length' has been deprecated; please use 'distributed.admin.low-level-log-length' instead
  warnings.warn(
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:237: FutureWarning: extra has been renamed to worker_extra_args. You are still using it (even if only set to []; please also check config files). If you did not set worker_extra_args yet, extra will be respected for now, but it will be removed in a future release. If you already set worker_extra_args, extra is ignored and you can remove it.
  w

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: http://10.148.0.219:8787/status,
Dashboard: http://10.148.0.219:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.148.0.219:52451,Workers: 0
Dashboard: http://10.148.0.219:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [7]:
cluster.close()

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:237: FutureWarning: extra has been renamed to worker_extra_args. You are still using it (even if only set to []; please also check config files). If you did not set worker_extra_args yet, extra will be respected for now, but it will be removed in a future release. If you already set worker_extra_args, extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home1/datahome/mdemol/

In [4]:
dfs

,,index,cycle_number,file,time,start_time_cut,end_time_cut
day,pass_number,,,,,,
2023-04-01,3,0,478,/home/datawork-lops-oc/aponte/swot/cswot/swot/...,2023-04-01 23:42:32.696536000,2023-04-01 11:47:13.760581216,2023-04-02 11:37:51.561104544
2023-04-02,3,1,479,/home/datawork-lops-oc/aponte/swot/cswot/swot/...,2023-04-02 23:33:10.425673088,2023-04-02 11:37:51.561104544,2023-04-03 11:28:29.307429408
2023-04-03,3,2,480,/home/datawork-lops-oc/aponte/swot/cswot/swot/...,2023-04-03 23:23:48.189185728,2023-04-03 11:28:29.307429408,2023-04-04 11:19:07.046608544
2023-04-04,3,3,481,/home/datawork-lops-oc/aponte/swot/cswot/swot/...,2023-04-04 23:14:25.904031360,2023-04-04 11:19:07.046608544,2023-04-05 11:09:44.780262144
2023-04-05,3,4,482,/home/datawork-lops-oc/aponte/swot/cswot/swot/...,2023-04-05 23:05:03.656492928,2023-04-05 11:09:44.780262144,2023-04-06 11:00:22.763027360
...,...,...,...,...,...,...,...
2023-07-05,16,180,573,/home/datawork-lops-oc/aponte/swot/cswot/swot/...,2023-07-05 19:33:50.331633280,2023-07-05 07:38:31.368841152,2023-07-06 07:29:09.302501440
2023-07-06,16,181,574,/home/datawork-lops-oc/aponte/swot/cswot/swot/...,2023-07-06 19:24:28.273369600,2023-07-06 07:29:09.302501440,2023-07-07 07:19:47.209324384
2023-07-07,16,182,575,/home/datawork-lops-oc/aponte/swot/cswot/swot/...,2023-07-07 19:15:06.145279168,2023-07-07 07:19:47.209324384,2023-07-08 07:10:25.074586176


___________
# CHOOSE

In [3]:
dfs = browse_swot_250m()
dsf = dfs.reset_index().to_xarray().chunk({'index':1})

In [4]:
def store_gaussian_250m(dsf, cutoff):
    
    def compute_gaussian_250m_one_file(da, cutoff):
        f = da['file'].values[0]
        from altimetry_filter import filter_diff_one
        method = 'gaussian'
        dg = filter_diff_one(f, filter_diff_method = method, filter_diff_kwargs = {'cutoff': cutoff})
        dg.to_netcdf(os.path.join(zarr_dir, 'before_coloc','preprocessed_swot','swot250m', f'{method}_{int(cutoff)}', f.split('/')[-1].replace('zarr', 'nc')))
        return da['file']
        
    template = dsf.file.isel(index = 0).compute()
    template = template.expand_dims({'index':dsf.index}).chunk({**dsf.chunks})
    
    d = dsf.map_blocks(compute_gaussian_250m_one_file, kwargs = {'cutoff':cutoff}, template=template).compute()
    return d
    
def store_diff_only_250m(dsf):
    
    def compute_diff_only_250m_one_file(da):
        f = da['file'].values[0]
        from altimetry_filter import filter_diff_one
        method = 'diff_only'
        try : 
            dg = filter_diff_one(f, filter_diff_method = method, filter_diff_kwargs = {})
        except: assert False, f
        dg.to_netcdf(os.path.join(zarr_dir, 'before_coloc','preprocessed_swot','swot250m', f'{method}', f.split('/')[-1].replace('zarr', 'nc')))
        return da['file']
        
    template = dsf.file.isel(index = 0).compute()
    template = template.expand_dims({'index':dsf.index}).chunk({**dsf.chunks})
    
    d = dsf.map_blocks(compute_diff_only_250m_one_file, template=template).compute()
    return d


In [11]:
d = store_diff_only_250m(dsf)

In [6]:
D = []
for cutoff in np.arange(35e3, 60e3, 5e3):
    D.append(store_gaussian_250m(dsf, cutoff))
    print(cutoff)

35000.0
40000.0
45000.0
50000.0
55000.0


In [5]:
variables = ['ancillary_surface_classification_flag',
             'cross_track_distance',
             'cvl_distance_to_coast',
             'duacs_editing_flag',
             'duacs_phase_screen',
             'duacs_phase_screen_orbit',
             'duacs_phase_screen_static',
             'duacs_relative_vorticity',
             'duacs_speed_meridional',
             'duacs_speed_meridional_abs',
             'duacs_speed_zonal',
             'duacs_speed_zonal_abs',
             'duacs_strain',
             'duacs_xcal',
             'sig0_karin_2',
             'phi',
             'cvl_swh_model', 
             'cycle_number'
            ]


def store_general_250m(dsf):
    
    def compute_general_250m_one_file(da):
        f = da['file'].values[0]
        method = 'general'
        try : 
            dg = xr.open_dataset(f)
            dg = add_grid_metrics(dg)[variables+['longitude', 'latitude']].rename({'cvl_swh_model':'swh_model', 'cvl_distance_to_coast':'distance_to_coast'})
        except: assert False, f
        dg.to_netcdf(os.path.join(zarr_dir, 'before_coloc','preprocessed_swot','swot250m', f'{method}', f.split('/')[-1].replace('zarr', 'nc')))
        return da['file']
        
    template = dsf.file.isel(index = 0).compute()
    template = template.expand_dims({'index':dsf.index}).chunk({**dsf.chunks})
    
    d = dsf.map_blocks(compute_general_250m_one_file, template=template).compute()
    return d

In [6]:
d = store_general_250m(dsf)